# Move legacy processed Neon results into `Neon/1`

This notebook migrates entries stored directly in each processed `Neon` directory:

```text
FLIC_<subject>/<activity>/Neon/<existing entries>
```

into the numbered layout expected by the current preprocessing pipeline:

```text
FLIC_<subject>/<activity>/Neon/1/<existing entries>
```

Existing numerical recording directories are left unchanged. If an individual destination already exists beneath `Neon/1`, that entry is skipped and neither copy is overwritten. The notebook starts in preview-only mode.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import re
import shutil

PROCESSING_DATASET_DIR = Path("/Volumes/FLIC_processing/NEWscriptedIndoorOutdoorVideos2026")

# Leave False for the first run. Change to True only after reviewing the plan.
APPLY_MOVES = False

# Optional filters. Empty sets mean all subjects and all activities.
SUBJECTS_TO_PROCESS: set[int] = set()
SUBJECTS_TO_SKIP: set[int] = set()
ACTIVITIES_TO_PROCESS: set[str] = set()
ACTIVITIES_TO_SKIP: set[str] = set()

In [ ]:
@dataclass(frozen=True)
class NeonMove:
    source: Path
    destination: Path


def is_selected(item, items_to_process, items_to_skip) -> bool:
    return item in items_to_process if items_to_process else item not in items_to_skip


def build_neon_move_plan(processing_root: Path):
    processing_root = processing_root.expanduser().resolve()
    if not processing_root.is_dir():
        raise NotADirectoryError(f"Processing dataset root is not a directory: {processing_root}")

    moves: list[NeonMove] = []
    skipped_existing: list[NeonMove] = []
    subject_dirs = sorted(
        (path for path in processing_root.iterdir()
         if path.is_dir() and re.fullmatch(r"FLIC_\d+", path.name)),
        key=lambda path: int(path.name.removeprefix("FLIC_")),
    )

    for subject_dir in subject_dirs:
        subject_number = int(subject_dir.name.removeprefix("FLIC_"))
        if not is_selected(subject_number, SUBJECTS_TO_PROCESS, SUBJECTS_TO_SKIP):
            continue

        activity_dirs = sorted((p for p in subject_dir.iterdir() if p.is_dir()), key=lambda p: p.name)
        for activity_dir in activity_dirs:
            if not is_selected(activity_dir.name, ACTIVITIES_TO_PROCESS, ACTIVITIES_TO_SKIP):
                continue

            neon_dir = activity_dir / "Neon"
            if not neon_dir.is_dir():
                continue

            # Numeric directories already represent individual recordings.
            legacy_entries = sorted(
                (entry for entry in neon_dir.iterdir()
                 if not (entry.is_dir() and entry.name.isdigit())),
                key=lambda path: path.name,
            )
            for source in legacy_entries:
                destination = neon_dir / "1" / source.name
                move = NeonMove(source=source, destination=destination)
                if destination.exists():
                    skipped_existing.append(move)
                else:
                    moves.append(move)

    return moves, skipped_existing


def print_neon_move_plan(moves: list[NeonMove], skipped_existing: list[NeonMove]) -> None:
    print(f"Legacy Neon entries ready to move: {len(moves)}")
    for move in moves:
        print(f"  {move.source} -> {move.destination}")

    print(f"Existing destinations to skip: {len(skipped_existing)}")
    for move in skipped_existing:
        print(f"  SKIP {move.source} (destination exists: {move.destination})")

## Preview the migration

Review the planned moves and skipped entries before enabling `APPLY_MOVES`.

In [ ]:
moves, skipped_existing = build_neon_move_plan(PROCESSING_DATASET_DIR)
print_neon_move_plan(moves, skipped_existing)

## Apply the migration

The plan is rebuilt immediately before execution. Each destination is checked again before its source is moved, and existing destinations are skipped without raising an error.

In [ ]:
def apply_neon_moves(moves: list[NeonMove], apply_moves: bool = False) -> None:
    if not apply_moves:
        print("PREVIEW ONLY: nothing was moved. Set APPLY_MOVES = True and rerun this cell to apply.")
        return

    moved_count = 0
    skipped_count = 0
    for move in moves:
        if not move.source.exists():
            print(f"SKIP missing source: {move.source}")
            skipped_count += 1
            continue
        if move.destination.exists():
            print(f"SKIP existing destination: {move.destination}")
            skipped_count += 1
            continue

        move.destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(move.source), str(move.destination))
        print(f"Moved: {move.source} -> {move.destination}")
        moved_count += 1

    print(f"Finished: moved {moved_count}; skipped {skipped_count}.")


moves, skipped_existing = build_neon_move_plan(PROCESSING_DATASET_DIR)
print_neon_move_plan(moves, skipped_existing)
apply_neon_moves(moves, apply_moves=APPLY_MOVES)

## Verify the resulting layout

After applying the migration, this reports any entries that still remain directly beneath selected processed `Neon` directories. Entries skipped because their destination already existed will remain in this list.

In [ ]:
def verify_neon_layout(processing_root: Path) -> None:
    processing_root = processing_root.expanduser().resolve()
    remaining_legacy: list[Path] = []
    numbered_recording_dirs: list[Path] = []

    for subject_dir in processing_root.iterdir():
        if not subject_dir.is_dir() or not re.fullmatch(r"FLIC_\d+", subject_dir.name):
            continue
        subject_number = int(subject_dir.name.removeprefix("FLIC_"))
        if not is_selected(subject_number, SUBJECTS_TO_PROCESS, SUBJECTS_TO_SKIP):
            continue
        for activity_dir in (path for path in subject_dir.iterdir() if path.is_dir()):
            if not is_selected(activity_dir.name, ACTIVITIES_TO_PROCESS, ACTIVITIES_TO_SKIP):
                continue
            neon_dir = activity_dir / "Neon"
            if not neon_dir.is_dir():
                continue
            for entry in neon_dir.iterdir():
                if entry.is_dir() and entry.name.isdigit():
                    numbered_recording_dirs.append(entry)
                else:
                    remaining_legacy.append(entry)

    print(f"Numbered processed Neon recording directories: {len(numbered_recording_dirs)}")
    print(f"Entries remaining directly beneath Neon: {len(remaining_legacy)}")
    for path in sorted(remaining_legacy):
        print(f"  {path}")


verify_neon_layout(PROCESSING_DATASET_DIR)